In [1]:
import dask.dataframe as dd
import pandas as pd
import numpy as np
import os

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Some globals

PROJECT_PATH  = '/content/drive/MyDrive/CENG574_PROJECT'
DATA_PATH = os.path.join(PROJECT_PATH, 'labeled_data')

INPUT_PARQUET = os.path.join(DATA_PATH, "cleaned_partitions")
OUTPUT_PARQUET = os.path.join(DATA_PATH, "nyc_taxi_grid_sampled.parquet")



In [4]:

# =====================================================
# PARAMETERS
# =====================================================

N_GRID = 50      # NxN spatial grid
MAX_PER_CELL = 10

# =====================================================
# NYC BOUNDING BOX
# =====================================================

LAT_MIN = 40.5774
LAT_MAX = 40.9176

LON_MIN = -74.15
LON_MAX = -73.7004



In [6]:
# =====================================================
# LOAD
# =====================================================

print("Loading merged parquet...")
USE_DASK= True
try:
  print("Reading using dask...")
  ddf = dd.read_parquet(INPUT_PARQUET)
except:
  USE_DASK= False
  print("Reading using pandas...")
  ddf = pd.read_parquet(INPUT_PARQUET)

print("Computing sampled dataframe...")

# Convert to pandas
if USE_DASK:
  df = ddf.compute()
else:
  df =ddf

print(f"Original samples: {len(df):,}")

Loading merged parquet...
Reading using dask...
Computing sampled dataframe...
Original samples: 12,367,735


In [7]:


# =====================================================
# FILTER TO NYC BOUNDS
# =====================================================

df = df[
    (df["pickup_latitude"] >= LAT_MIN) &
    (df["pickup_latitude"] <= LAT_MAX) &
    (df["pickup_longitude"] >= LON_MIN) &
    (df["pickup_longitude"] <= LON_MAX)
]

print(f"After coordinate filtering: {len(df):,}")

# =====================================================
# GRID ASSIGNMENT
# =====================================================

lat_step = (LAT_MAX - LAT_MIN) / N_GRID
lon_step = (LON_MAX - LON_MIN) / N_GRID

df["grid_y"] = (
    ((df["pickup_latitude"] - LAT_MIN) / lat_step)
    .astype(int)
    .clip(0, N_GRID - 1)
)

df["grid_x"] = (
    ((df["pickup_longitude"] - LON_MIN) / lon_step)
    .astype(int)
    .clip(0, N_GRID - 1)
)


After coordinate filtering: 12,367,735


In [8]:

# =====================================================
# SAMPLE EACH CELL
# =====================================================

sampled_parts = []

for _, group in df.groupby(["grid_x", "grid_y"]):

    if len(group) > MAX_PER_CELL:
        sampled_parts.append(
            group.sample(MAX_PER_CELL, random_state=42)
        )
    else:
        sampled_parts.append(group)

sampled_df = pd.concat(sampled_parts, ignore_index=True)

# Remove helper columns
sampled_df = sampled_df.drop(
    columns=["grid_x", "grid_y"]
)

print(f"Final sampled size: {len(sampled_df):,}")
print(f"Final data set features (might include ground truth): {sampled_df.shape[1]}")

Final sampled size: 8,716
Final data set features (might include ground truth): 16


In [9]:

# =====================================================
# SAVE
# =====================================================

sampled_df.to_parquet(
    OUTPUT_PARQUET,
    index=False
)

print(f"Saved to {OUTPUT_PARQUET}")

Saved to /content/drive/MyDrive/CENG574_PROJECT/labeled_data/nyc_taxi_grid_sampled.parquet
